<h1><font color='blue'>Análise de Sentimentos e Sistemas de Recomendação</font></h1>
<h2><b>Prática U2-T3-V1.2-Recomendação baseada em conteúdo</b></h2>


---

# Exemplo 1 - Recomendação baseada em conteúdo (não personalizada)

## Lendo o dataset de filmes

In [22]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [23]:
import pandas as pd
import numpy as np

movies = pd.read_csv('https://drive.google.com/uc?export=download&id=1yjGdaAJ5-YgIUx_1mLTcPqiZ8OI9TeqH', encoding='latin-1', low_memory=False)
movies['tags'] = movies['tags'].apply(lambda x: '' if isinstance(x,float) else x)
movies.set_index('movieId', inplace=True)
movies.head()

,title,genres,tags,tag_count,vote_count,vote_average,imdb_score
movieId,,,,,,,
1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,fun pixar,2,215,3.920930,3.894473
2,Jumanji,Adventure|Children|Fantasy,fantasy magic_board_game game robin_williams,4,110,3.431818,3.419009
3,Grumpier Old Men,Comedy|Romance,old moldy,2,52,3.259615,3.260033
4,Waiting to Exhale,Comedy|Drama|Romance,,0,7,2.357143,2.866377
5,Father of the Bride Part II,Comedy,pregnancy remake,2,49,3.071429,3.101070


Formatando a lista de documentos

In [24]:
# Baixar recursos do NLTK
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

Criar função para tokenizar, normalizar para minúsculos e remover caracteres especiais

In [25]:
import re
from nltk.tokenize import word_tokenize
import unicodedata

def preprocess(text):
    text = text.lower()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8")
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    return tokens


In [26]:
def retorna_documentos(df):
    documentos = []
    for title, genres, tags in df[['title', 'genres', 'tags']].values:
        lista = genres.split('|')
        documentos.append(' '.join(preprocess(title.lower() + ' ' + ' '.join(lista).lower() + ' ' + tags)))
    return documentos

documentos = retorna_documentos(movies)
documentos[:5]

['toy story adventure animation children comedy fantasy fun pixar',
 'jumanji adventure children fantasy fantasy magic board game game robin williams',
 'grumpier old men comedy romance old moldy',
 'waiting to exhale comedy drama romance',
 'father of the bride part ii comedy pregnancy remake']

## Representando filmes como *embeddings*

### Criando os embeddings dos documentos

Vamos usar uma implementação prática da estratégia **SBERT (Sentence-BERT)** para a geração dos *embeddings*. O **SBERT** é uma modificação do BERT treinada para gerar embeddings de frases/documentos que podem ser comparados diretamente via similaridade de cosseno.

In [27]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Converter documentos para *embeddings*

In [28]:
embeddings = model.encode(documentos, normalize_embeddings=True)
print(embeddings.shape)

(9724, 384)


In [29]:
embeddings[0]

array([-5.19503318e-02,  8.96416837e-04,  1.36910640e-02, -1.78741980e-02,
       -5.73252290e-02,  1.23114204e-02,  1.00211255e-01, -2.27374006e-02,
        1.65828094e-02,  6.23797299e-03,  1.19955409e-02, -1.23948352e-02,
        2.36282474e-03,  7.46658817e-02,  1.62983965e-02,  1.64942443e-02,
        4.44534831e-02,  4.08788361e-02,  9.35138389e-02, -7.51637369e-02,
        6.62237704e-02,  3.65036912e-03,  6.79250360e-02, -2.06108484e-02,
       -5.57867959e-02,  4.09747213e-02, -3.15521075e-03, -3.03457137e-02,
       -3.62760499e-02, -4.11853157e-02,  7.65422583e-02,  5.33682480e-02,
        1.81106683e-02, -3.64747569e-02,  1.25191836e-02,  4.91976365e-03,
        2.17597671e-02, -5.34348153e-02, -2.04926953e-02,  2.52218004e-02,
       -5.78783303e-02,  1.00279320e-02, -9.96980444e-03, -8.22473019e-02,
        5.69721917e-03, -5.05874911e-03, -3.84973958e-02, -8.74377191e-02,
        5.57347164e-02,  8.74505565e-02, -4.46493141e-02, -2.20414549e-02,
        2.30034553e-02,  

## Recomendação de itens baseada em conteúdo

A partir dos *embeddings* gerados para cada documento, será possível executar consultas recuperando os documentos relevantes, de acordo com palavras chaves informadas pelo usuário.

Vamos usar a biblioteca **`hnswlib`** para executar uma busca vetorial rápida e eficiente baseada em grafos HNSW.

É muito usada em busca semântica com *embeddings* para encontrar os documentos mais similares em grandes coleções.

In [30]:
!pip install -qU hnswlib

Preparando o índice para a busca

In [31]:
import hnswlib

index = hnswlib.Index(space='cosine', dim=embeddings.shape[1])
index.init_index(max_elements=len(documentos))
index.add_items(embeddings)

### Encontrar filmes similares

In [32]:
ids = [1, 108932]
retorna_documentos(movies.query('movieId in ' + str(ids)))

['toy story adventure animation children comedy fantasy fun pixar',
 'the lego movie action adventure animation children comedy fantasy colorful feel good quirky fun clever cheeky imaginative']

Encontra os *embeddings* para os filmes

In [33]:
embeddings_query = model.encode(retorna_documentos(movies.query('movieId in ' + str(ids))))
embeddings_query


array([[-5.19503169e-02,  8.96431389e-04,  1.36910751e-02,
        -1.78742185e-02, -5.73252477e-02,  1.23114372e-02,
         1.00211255e-01, -2.27373838e-02,  1.65828001e-02,
         6.23796228e-03,  1.19955372e-02, -1.23948390e-02,
         2.36285292e-03,  7.46658742e-02,  1.62983779e-02,
         1.64942387e-02,  4.44534793e-02,  4.08788174e-02,
         9.35138315e-02, -7.51637891e-02,  6.62238002e-02,
         3.65038449e-03,  6.79250658e-02, -2.06108484e-02,
        -5.57868108e-02,  4.09747101e-02, -3.15523404e-03,
        -3.03457491e-02, -3.62760425e-02, -4.11853381e-02,
         7.65422508e-02,  5.33682778e-02,  1.81106627e-02,
        -3.64747755e-02,  1.25192199e-02,  4.91975714e-03,
         2.17597745e-02, -5.34348302e-02, -2.04926934e-02,
         2.52218265e-02, -5.78783229e-02,  1.00279097e-02,
        -9.96984169e-03, -8.22473168e-02,  5.69725130e-03,
        -5.05870767e-03, -3.84973623e-02, -8.74377117e-02,
         5.57346977e-02,  8.74505341e-02, -4.46493402e-0

Faz a média dos *embeddings* para encontrar os filmes similares

In [34]:
np.mean(embeddings_query, axis=0)

array([-6.92324638e-02,  6.92200731e-04,  1.97593383e-02, -2.16796994e-02,
       -4.88054268e-02, -1.51488278e-03,  1.00793526e-01, -1.20748300e-02,
        3.53812892e-03, -6.62334682e-03, -2.60515418e-03, -1.70853138e-02,
       -1.89814232e-02,  7.08111599e-02,  1.10414475e-02,  2.13854164e-02,
        6.64860606e-02,  3.45265716e-02,  7.45241418e-02, -6.98116124e-02,
        4.76667173e-02, -9.99997091e-03,  6.96889758e-02, -1.06576607e-02,
       -5.46349958e-02,  2.67314985e-02,  7.54536921e-03, -2.26662047e-02,
       -5.29745072e-02, -2.60952730e-02,  7.00346902e-02,  5.89012802e-02,
        9.74311866e-03, -3.43117118e-02,  2.60102004e-02,  3.14724520e-02,
        2.66602542e-02, -3.48419361e-02, -3.13553847e-02, -2.04935297e-03,
       -6.54895157e-02,  1.19180093e-02, -1.58172995e-02, -6.51483759e-02,
        1.60977785e-02,  4.29980410e-03, -1.68463979e-02, -9.07835364e-02,
        3.05366702e-02,  6.14875369e-02, -4.26211357e-02, -2.12162584e-02,
        8.19272641e-03, -

### Função para retornar a recomendação baseada em conteúdo

In [35]:
def content_based_recommendation(movie_ids=[], topN=10):

    ranking = []
    embeddings_query = model.encode(retorna_documentos(movies.query('movieId in ' + str(movie_ids))))
    if len(movie_ids) > 1: embeddings_query = np.mean(embeddings_query, axis=0)

    labels, distances = index.knn_query(embeddings_query, k=topN*2)
    for idx,(doc_id, dist) in enumerate(zip(labels[0], distances[0])):
        if (movies.index.values[doc_id] not in movie_ids):
            ranking.append( list(movies.iloc[doc_id][['title', 'genres', 'tags', 'imdb_score']].values) + [idx, 1-dist] )

    aux = pd.DataFrame(ranking, columns=['title', 'genres', 'tags', 'imdb_score', 'rank_pos', 'rank_score'])
    return aux.sort_values(by=['rank_score', 'imdb_score'], ascending=False).head(topN)

content_based_recommendation([1])

,title,genres,tags,imdb_score,rank_pos,rank_score
0,Toy Story 3,Adventure|Animation|Children|Comedy|Fantasy|IMAX,,3.990032,1,0.853986
1,Toy Story 2,Adventure|Animation|Children|Comedy|Fantasy,tom_hanks original pixar animation disney funn...,3.810019,2,0.847664
2,The Lego Movie,Action|Adventure|Animation|Children|Comedy|Fan...,colorful feel-good quirky fun clever cheeky im...,3.734051,3,0.837530
3,Turbo,Adventure|Animation|Children|Comedy|Fantasy,,3.186203,4,0.821473
4,"Wild, The",Adventure|Animation|Children|Comedy|Fantasy,,3.186203,5,0.815881
5,"Bug's Life, A",Adventure|Animation|Children|Comedy,pixar,3.493684,6,0.814950
6,Antz,Adventure|Animation|Children|Comedy|Fantasy,,3.247445,7,0.810395
7,"Monsters, Inc.",Adventure|Animation|Children|Comedy|Fantasy,,3.832355,8,0.801546
8,Toys,Comedy|Fantasy,,2.650415,9,0.791271
9,Robots,Adventure|Animation|Children|Comedy|Fantasy|Sc...,,3.102216,10,0.789724


In [36]:
ids = [1, 108932]
retorna_documentos(movies.query('movieId in ' + str(ids)))

['toy story adventure animation children comedy fantasy fun pixar',
 'the lego movie action adventure animation children comedy fantasy colorful feel good quirky fun clever cheeky imaginative']

In [37]:
content_based_recommendation([1, 108932])

,title,genres,tags,imdb_score,rank_pos,rank_score
0,Turbo,Adventure|Animation|Children|Comedy|Fantasy,,3.186203,2,0.855695
1,Toy Story 3,Adventure|Animation|Children|Comedy|Fantasy|IMAX,,3.990032,3,0.851180
2,"Wild, The",Adventure|Animation|Children|Comedy|Fantasy,,3.186203,4,0.842802
3,Antz,Adventure|Animation|Children|Comedy|Fantasy,,3.247445,5,0.842397
4,"Monsters, Inc.",Adventure|Animation|Children|Comedy|Fantasy,,3.832355,6,0.822548
5,Toy Story 2,Adventure|Animation|Children|Comedy|Fantasy,tom_hanks original pixar animation disney funn...,3.810019,7,0.821488
6,Home,Adventure|Animation|Children|Comedy|Fantasy|Sc...,,3.256590,8,0.815789
7,Ice Age,Adventure|Animation|Children|Comedy,,3.647468,9,0.808273
8,"Bug's Life, A",Adventure|Animation|Children|Comedy,pixar,3.493684,10,0.808222
9,Robots,Adventure|Animation|Children|Comedy|Fantasy|Sc...,,3.102216,11,0.805389


In [38]:
ids = [858]
retorna_documentos(movies.query('movieId in ' + str(ids)))

['godfather the crime drama mafia']

In [39]:
content_based_recommendation([858])

,title,genres,tags,imdb_score,rank_pos,rank_score
0,"Godfather: Part II, The",Crime|Drama,al_pacino mafia,4.194652,1,0.821600
1,Casino,Crime|Drama,mafia,3.861121,2,0.768495
2,Donnie Brasco,Crime|Drama,mafia,3.669869,3,0.749043
3,Gangster Squad,Action|Crime|Drama,,3.280170,4,0.745306
4,"Godfather: Part III, The",Crime|Drama|Mystery|Thriller,mafia classic andy_garcia francis_ford_coppola...,3.340038,5,0.736036
5,Goodfellas,Crime|Drama,mafia,4.184163,6,0.735513
6,Carlito's Way,Crime|Drama,al_pacino gangster mafia,3.654158,7,0.692946
7,Miller's Crossing,Crime|Drama|Film-Noir|Thriller,mafia,3.926277,8,0.680292
8,Scarface,Crime|Drama,,3.351094,9,0.679608
9,Gangster No. 1,Action|Crime|Thriller,,3.196836,10,0.678310
